# Module 10.5: Creative Image Agent — Learning to Paint with RL

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FlashVision/VisionRL/blob/main/Module_10_RL_For_Image_Generation/05_Creative_Image_Agent/creative_image_agent.ipynb)

---

## Learning Objectives

By the end of this notebook, you will:

1. **Formulate** sequential painting as a Markov Decision Process
2. **Implement** a canvas environment with brush stroke actions
3. **Train** a REINFORCE agent that learns to paint target images
4. **Visualize** step-by-step painting progression like an animation
5. **Explore** how different reward weightings produce different "painting styles"

---

## Prerequisites

- Understanding of REINFORCE algorithm
- Basic image processing (pixels, colors)
- PyTorch fundamentals

In [ ]:
!pip install torch torchvision numpy matplotlib scikit-image

In [ ]:
# Download REAL open-source dataset for image generation (TINY - under 200MB)
import torchvision
import torchvision.transforms as transforms

# MNIST for GAN training (classic, tiny, real)
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize([0.5], [0.5])])
mnist = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
print(f"✅ MNIST: {len(mnist)} real images for GAN training (11MB)")

# FashionMNIST for more complex generation
fashion = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
print(f"✅ FashionMNIST: {len(fashion)} real clothing images (30MB)")

# CIFAR-10 for color image generation
transform_color = transforms.Compose([transforms.ToTensor(), transforms.Normalize([0.5]*3, [0.5]*3)])
cifar10 = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_color)
print(f"✅ CIFAR-10: {len(cifar10)} real color photos for generation (170MB)")

print("\n📦 Total download: ~211MB")
print("   NO CelebA needed - MNIST/FashionMNIST/CIFAR-10 are perfect for learning!")
print("   MNIST → simple GAN, FashionMNIST → medium, CIFAR-10 → color generation")

## Deep Derivation: Creativity, Novelty Search, and RL for Artistic Generation

### Step 1: Computational Creativity as Optimization
Define creativity as the combination of novelty and quality:
$$\text{Creativity}(x) = \underbrace{Q(x)}_{\text{quality/aesthetics}} \cdot \underbrace{N(x)}_{\text{novelty}}$$

**Novelty measure (distance to nearest neighbor in archive $\mathcal{A}$):**
$$N(x) = \frac{1}{k}\sum_{i=1}^k \|f(x) - f(x_i^{\text{nn}})\|_2$$

where $x_i^{\text{nn}}$ are the $k$-nearest neighbors in feature space $f$.

**Derivation:** This is the kernel density estimate of surprise:
$$N(x) \propto \frac{1}{\hat{p}(x)}$$

where $\hat{p}(x) = \frac{k}{n \cdot V_k(x)}$ is the $k$-NN density estimate and $V_k(x)$ is the volume of the ball containing $k$ neighbors. High novelty = low density = far from existing examples. $\blacksquare$

### Step 2: Aesthetic Quality Score (NIMA Architecture)
The Neural Image Assessment model predicts aesthetic score distribution:
$$p(\text{score} = s | I) = \text{softmax}(W \cdot \text{CNN}(I) + b), \quad s \in \{1, 2, \ldots, 10\}$$

**Expected aesthetic score:**
$$\mu(I) = \sum_{s=1}^{10} s \cdot p(s|I)$$

**Earth Mover's Distance (EMD) loss for ordinal regression:**
$$\text{EMD}(p, \hat{p}) = \left(\frac{1}{N}\sum_{k=1}^N \left|\text{CDF}_p(k) - \text{CDF}_{\hat{p}}(k)\right|^r\right)^{1/r}$$

where $\text{CDF}_p(k) = \sum_{s=1}^k p(s)$.

**Why EMD instead of cross-entropy?** Cross-entropy treats scores as categorical (predicting 5 vs 6 is penalized the same as 5 vs 1). EMD respects the ordinal structure — nearby scores incur less penalty than distant scores.

**Formal proof:** EMD is a metric on probability distributions over ordered sets. By Kantorovich-Rubinstein duality:
$$\text{EMD}_1(p, q) = \sup_{\|f\|_L \leq 1} E_p[f] - E_q[f]$$

where the supremum is over 1-Lipschitz functions. $\blacksquare$

### Step 3: Novelty Search via Quality-Diversity (MAP-Elites)
MAP-Elites maintains a grid of "niches" indexed by behavior descriptors $\mathbf{b}(x) \in \mathbb{R}^d$:

For each new solution $x$:
1. Compute descriptor: $\mathbf{b}(x) = (\text{color\_histogram}(x), \text{texture\_energy}(x), \ldots)$
2. Find cell: $c = \text{discretize}(\mathbf{b}(x))$
3. If cell empty or $Q(x) > Q(\text{archive}[c])$: $\text{archive}[c] \leftarrow x$

**Derivation of coverage guarantee:** With a $d$-dimensional behavior space discretized into $n$ bins per dimension, there are $n^d$ cells. After $T$ evaluations, the expected coverage:
$$E[\text{coverage}] = n^d \left(1 - \left(1 - \frac{1}{n^d}\right)^T\right) \approx n^d(1 - e^{-T/n^d})$$

For 50% coverage: $T \approx n^d \ln 2$. This grows exponentially with descriptor dimension, motivating low-dimensional descriptors. $\blacksquare$

### Step 4: Intrinsic Motivation for Exploration
**Random Network Distillation (RND):**
$$r_{\text{intrinsic}}(s) = \|f_\theta(s) - f_{\text{fixed}}(s)\|^2$$

where $f_{\text{fixed}}$ is a randomly initialized, frozen network and $f_\theta$ is trained to predict $f_{\text{fixed}}$'s output.

**Derivation of why this measures novelty:** The prediction error $\|f_\theta(s) - f_{\text{fixed}}(s)\|^2$ is high for novel states because $f_\theta$ hasn't been trained on similar inputs.

**Formal bound:** After $n$ visits to state $s$, the expected RND bonus:
$$E[r_{\text{RND}}(s)] \leq \frac{C}{n+1}$$

for some constant $C$ depending on the network architecture. This provides an $O(1/n)$ decay, similar to the optimal UCB exploration bonus. $\blacksquare$

### Step 5: Stroke-Based Rendering Policy
The agent generates art through sequential brush strokes:
$$a_t = (x, y, x', y', r_0, r_1, \theta, R, G, B, \alpha) \in \mathbb{R}^{11}$$

**Bezier curve parameterization:** Each stroke follows a quadratic Bezier:
$$B(t) = (1-t)^2 P_0 + 2(1-t)t P_1 + t^2 P_2, \quad t \in [0,1]$$

with varying radius $r(t) = (1-t)r_0 + t \cdot r_1$.

**Differentiable rendering:** To enable gradient-based training, the stroke must be differentiable. Use a soft rasterizer:
$$I_{\text{stroke}}(p) = \alpha \cdot \text{Color} \cdot \exp\left(-\frac{d(p, B)^2}{2r(t_p)^2}\right)$$

where $d(p, B)$ is the distance from pixel $p$ to the nearest point on the Bezier curve. This Gaussian falloff makes the rendering smooth and differentiable. $\blacksquare$

### Step 6: Reinforcement Learning Objective for Creative Agents
The creative RL objective balances quality, novelty, and constraint satisfaction:
$$J(\pi) = E_{\pi}\left[\sum_{t=0}^T \gamma^t \left(\underbrace{w_1 R_{\text{aesthetic}}}_{\text{beauty}} + \underbrace{w_2 R_{\text{novelty}}}_{\text{originality}} + \underbrace{w_3 R_{\text{semantic}}}_{\text{meaning}} - \underbrace{w_4 C_{\text{budget}}}_{\text{efficiency}}\right)\right]$$

where $C_{\text{budget}} = \mathbb{1}[t > T_{\max}]$ penalizes using too many strokes.

**Policy gradient with curiosity-driven exploration:**
$$\nabla_\theta J = E_\pi\left[\sum_t \nabla_\theta \log\pi_\theta(a_t|s_t) \cdot \left(\hat{A}_t^{\text{extrinsic}} + \beta \hat{A}_t^{\text{intrinsic}}\right)\right]$$

The intrinsic advantage $\hat{A}^{\text{intrinsic}}$ from RND encourages the agent to try novel brush strokes and color combinations.

### Step 7: Diversity Measurement via Maximum Mean Discrepancy
To evaluate whether the creative agent produces diverse outputs, use MMD:
$$\text{MMD}^2(\mathcal{P}, \mathcal{Q}) = E_{x,x'\sim\mathcal{P}}[k(x,x')] - 2E_{x\sim\mathcal{P}, y\sim\mathcal{Q}}[k(x,y)] + E_{y,y'\sim\mathcal{Q}}[k(y,y')]$$

with Gaussian kernel $k(x,y) = \exp(-\|f(x)-f(y)\|^2 / 2\sigma^2)$ in feature space.

**Derivation:** MMD measures the distance between distributions in a Reproducing Kernel Hilbert Space (RKHS). By the kernel mean embedding theorem:
$$\text{MMD}(\mathcal{P}, \mathcal{Q}) = \|\mu_\mathcal{P} - \mu_\mathcal{Q}\|_\mathcal{H}$$

where $\mu_\mathcal{P} = E_{x\sim\mathcal{P}}[\phi(x)]$ is the mean embedding. For characteristic kernels (e.g., Gaussian), $\text{MMD} = 0 \iff \mathcal{P} = \mathcal{Q}$.

**Finite-sample estimator (unbiased):**
$$\widehat{\text{MMD}}^2 = \frac{1}{n(n-1)}\sum_{i\neq j} k(x_i,x_j) - \frac{2}{nm}\sum_{i,j} k(x_i,y_j) + \frac{1}{m(m-1)}\sum_{i\neq j} k(y_i,y_j) \quad \blacksquare$$

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from collections import deque
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 1. Mathematical Foundation: Painting as an MDP

### 1.1 The Sequential Painting Problem

We model a digital painting agent that places brush strokes on a canvas to reproduce a target image. This is formulated as an MDP $(\mathcal{S}, \mathcal{A}, P, R, \gamma)$.

### 1.2 State Space

The state at step $t$ encodes:

$$s_t = \left(C^{(t)}, \; \phi(C^{(t)}), \; \phi(I_{\text{target}}), \; t/T\right)$$

where $C^{(t)} \in [0,1]^{3 \times H \times W}$ is the current canvas and $\phi$ extracts summary features.

In practice, we use a compact state:
$$s_t = \left[\text{MSE}(C^{(t)}, I), \; \mu(C^{(t)}), \; \sigma(C^{(t)}), \; \mu(I-C^{(t)}), \; t/T\right]$$

### 1.3 Action Space

Each action $a_t$ specifies a brush stroke parameterized by:

$$a_t = (x, y, r, g, b, s)$$

- **Position**: $(x, y) \in \{0, 1, \ldots, H-1\} \times \{0, 1, \ldots, W-1\}$
- **Color**: $(r, g, b) \in \{0, \frac{1}{K-1}, \ldots, 1\}^3$ (discretized)
- **Brush size**: $s \in \{1, 2, 3\}$ (radius in pixels)

We discretize and flatten this into a single action index. For a $16 \times 16$ canvas with $K=4$ color levels and 3 sizes:

$$|\mathcal{A}| = H \times W \times K^3 \times |S| = 16 \times 16 \times 64 \times 3 = 49152$$

This is too large, so we use a **factored action space** with separate heads.

### 1.4 Brush Stroke Model

A circular brush stroke applies color to a disk region:

$$C^{(t+1)}_{c,i,j} = \begin{cases}
\alpha \cdot \text{color}_c + (1-\alpha) \cdot C^{(t)}_{c,i,j} & \text{if } (i-y)^2 + (j-x)^2 \leq s^2 \\
C^{(t)}_{c,i,j} & \text{otherwise}
\end{cases}$$

where $\alpha \in (0, 1]$ is the brush opacity.

### 1.5 Reward Function

The reward balances similarity improvement against stroke efficiency:

$$r_t = \underbrace{\left(\text{MSE}(C^{(t-1)}, I) - \text{MSE}(C^{(t)}, I)\right)}_{\text{improvement}} \cdot \lambda_{\text{improve}} - \underbrace{\lambda_{\text{stroke}}}_{\text{stroke cost}}$$

Different $\lambda$ ratios produce different "styles":
- High $\lambda_{\text{improve}}$: detailed, many-stroke paintings
- High $\lambda_{\text{stroke}}$: minimalist, bold-stroke paintings

### 1.6 REINFORCE Update

$$\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}\left[\sum_{t=0}^{T-1} \nabla_\theta \log \pi_\theta(a_t|s_t) \left(G_t - b\right)\right]$$

where $G_t = \sum_{t'=t}^{T-1} \gamma^{t'-t} r_{t'}$ is the return and $b$ is a baseline.

## 2. Target Images

We create simple synthetic target images that the agent will learn to paint.

In [ ]:
CANVAS_SIZE = 16

def create_target_circle(size=CANVAS_SIZE):
    """Red circle on blue background."""
    img = np.full((3, size, size), 0.15, dtype=np.float32)
    img[2, :, :] = 0.4
    y, x = np.ogrid[:size, :size]
    mask = ((x - size//2)**2 + (y - size//2)**2) < (size//4)**2
    img[0][mask] = 0.9
    img[1][mask] = 0.15
    img[2][mask] = 0.15
    return img


def create_target_gradient(size=CANVAS_SIZE):
    """Smooth color gradient."""
    img = np.zeros((3, size, size), dtype=np.float32)
    for i in range(size):
        for j in range(size):
            img[0, i, j] = i / size
            img[1, i, j] = j / size
            img[2, i, j] = 1.0 - (i + j) / (2 * size)
    return img


def create_target_flag(size=CANVAS_SIZE):
    """Three horizontal color bands."""
    img = np.zeros((3, size, size), dtype=np.float32)
    h3 = size // 3
    img[0, :h3, :] = 0.9; img[1, :h3, :] = 0.2; img[2, :h3, :] = 0.1
    img[0, h3:2*h3, :] = 0.9; img[1, h3:2*h3, :] = 0.9; img[2, h3:2*h3, :] = 0.9
    img[0, 2*h3:, :] = 0.1; img[1, 2*h3:, :] = 0.5; img[2, 2*h3:, :] = 0.9
    return img


def create_target_cross(size=CANVAS_SIZE):
    """Green cross on dark background."""
    img = np.full((3, size, size), 0.1, dtype=np.float32)
    cx, cy = size // 2, size // 2
    thickness = max(2, size // 8)
    img[1, cy-thickness:cy+thickness, :] = 0.8
    img[1, :, cx-thickness:cx+thickness] = 0.8
    return img


targets = {
    'Red Circle': create_target_circle(),
    'Gradient': create_target_gradient(),
    'Flag': create_target_flag(),
    'Green Cross': create_target_cross()
}

fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for ax, (name, img) in zip(axes, targets.items()):
    ax.imshow(img.transpose(1, 2, 0), interpolation='nearest')
    ax.set_title(name, fontsize=12, fontweight='bold')
    ax.axis('off')
plt.suptitle('Target Images for the Painting Agent', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Canvas Environment

In [ ]:
class PaintingEnv:
    """RL environment for sequential painting."""
    
    N_POSITIONS = CANVAS_SIZE * CANVAS_SIZE  # 256
    COLOR_LEVELS = 4  # 0.0, 0.33, 0.67, 1.0
    BRUSH_SIZES = [1, 2, 3]
    BRUSH_ALPHA = 0.85
    
    def __init__(self, target, max_strokes=60, stroke_penalty=0.001, improve_scale=500.0):
        self.target = target.copy()
        self.max_strokes = max_strokes
        self.stroke_penalty = stroke_penalty
        self.improve_scale = improve_scale
        self.size = target.shape[1]
        
        self.color_values = np.linspace(0, 1, self.COLOR_LEVELS)
        self.reset()
    
    def reset(self):
        self.canvas = np.full((3, self.size, self.size), 0.5, dtype=np.float32)
        self.step_count = 0
        self.prev_mse = np.mean((self.canvas - self.target) ** 2)
        self.history = [self.canvas.copy()]
        self.action_history = []
        return self._get_state()
    
    def _get_state(self):
        diff = self.target - self.canvas
        mse = np.mean((self.canvas - self.target) ** 2)
        state = np.array([
            mse,
            self.canvas.mean(),
            self.canvas.std(),
            diff.mean(),
            np.abs(diff).mean(),
            diff[0].mean(), diff[1].mean(), diff[2].mean(),
            self.step_count / self.max_strokes
        ], dtype=np.float32)
        return state
    
    @property
    def state_dim(self):
        return 9
    
    def apply_stroke(self, pos_idx, r_idx, g_idx, b_idx, size_idx):
        """Apply a brush stroke to the canvas."""
        y = pos_idx // self.size
        x = pos_idx % self.size
        
        color = np.array([
            self.color_values[r_idx],
            self.color_values[g_idx],
            self.color_values[b_idx]
        ])
        radius = self.BRUSH_SIZES[size_idx]
        alpha = self.BRUSH_ALPHA
        
        yy, xx = np.ogrid[:self.size, :self.size]
        mask = ((xx - x)**2 + (yy - y)**2) <= radius**2
        
        for c in range(3):
            self.canvas[c][mask] = alpha * color[c] + (1 - alpha) * self.canvas[c][mask]
        
        self.canvas = np.clip(self.canvas, 0, 1)
    
    def step(self, action_dict):
        """Take one painting step.
        action_dict: {'pos': int, 'r': int, 'g': int, 'b': int, 'size': int}
        """
        self.apply_stroke(
            action_dict['pos'],
            action_dict['r'], action_dict['g'], action_dict['b'],
            action_dict['size']
        )
        self.step_count += 1
        
        current_mse = np.mean((self.canvas - self.target) ** 2)
        improvement = self.prev_mse - current_mse
        reward = improvement * self.improve_scale - self.stroke_penalty
        self.prev_mse = current_mse
        
        done = self.step_count >= self.max_strokes
        
        self.history.append(self.canvas.copy())
        self.action_history.append(action_dict)
        
        return self._get_state(), reward, done

print('PaintingEnv defined.')

## 4. Painting Agent (REINFORCE with Factored Actions)

The policy network has separate heads for position, color channels, and brush size.

In [ ]:
class PaintingAgent(nn.Module):
    """REINFORCE agent with factored action space for painting."""
    
    def __init__(self, state_dim=9, hidden_dim=128,
                 n_positions=CANVAS_SIZE*CANVAS_SIZE,
                 n_colors=4, n_sizes=3):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )
        
        self.pos_head = nn.Linear(hidden_dim, n_positions)
        self.r_head = nn.Linear(hidden_dim, n_colors)
        self.g_head = nn.Linear(hidden_dim, n_colors)
        self.b_head = nn.Linear(hidden_dim, n_colors)
        self.size_head = nn.Linear(hidden_dim, n_sizes)
        
        self.saved_log_probs = []
        self.rewards = []
    
    def forward(self, state):
        h = self.backbone(state)
        return {
            'pos': F.softmax(self.pos_head(h), dim=-1),
            'r': F.softmax(self.r_head(h), dim=-1),
            'g': F.softmax(self.g_head(h), dim=-1),
            'b': F.softmax(self.b_head(h), dim=-1),
            'size': F.softmax(self.size_head(h), dim=-1)
        }
    
    def select_action(self, state):
        state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
        probs = self.forward(state_t)
        
        action = {}
        log_prob = 0
        
        for key in ['pos', 'r', 'g', 'b', 'size']:
            dist = torch.distributions.Categorical(probs[key])
            a = dist.sample()
            log_prob = log_prob + dist.log_prob(a)
            action[key] = a.item()
        
        self.saved_log_probs.append(log_prob)
        return action


def update_painting_agent(agent, optimizer, gamma=0.99):
    """REINFORCE update with baseline subtraction."""
    R = 0
    returns = deque()
    for r in reversed(agent.rewards):
        R = r + gamma * R
        returns.appendleft(R)
    
    returns = torch.tensor(list(returns), dtype=torch.float32, device=device)
    if returns.std() > 1e-8:
        returns = (returns - returns.mean()) / (returns.std() + 1e-8)
    
    loss = sum(-lp * R for lp, R in zip(agent.saved_log_probs, returns))
    
    optimizer.zero_grad()
    loss.backward()
    nn.utils.clip_grad_norm_(agent.parameters(), 1.0)
    optimizer.step()
    
    agent.saved_log_probs.clear()
    agent.rewards.clear()
    return loss.item()

print('PaintingAgent defined.')

## 5. Training the Painting Agent

In [ ]:
def train_painter(target_img, target_name, n_episodes=300, max_strokes=50,
                  stroke_penalty=0.001, improve_scale=500.0, lr=1e-3):
    """Train a painting agent on a single target image."""
    env = PaintingEnv(target_img, max_strokes=max_strokes,
                      stroke_penalty=stroke_penalty, improve_scale=improve_scale)
    agent = PaintingAgent(state_dim=env.state_dim).to(device)
    optimizer = optim.Adam(agent.parameters(), lr=lr)
    
    episode_rewards = []
    episode_mses = []
    best_mse = float('inf')
    best_history = None
    
    for ep in range(n_episodes):
        state = env.reset()
        total_reward = 0
        done = False
        
        while not done:
            action = agent.select_action(state)
            state, reward, done = env.step(action)
            agent.rewards.append(reward)
            total_reward += reward
        
        update_painting_agent(agent, optimizer)
        
        final_mse = np.mean((env.canvas - target_img) ** 2)
        episode_rewards.append(total_reward)
        episode_mses.append(final_mse)
        
        if final_mse < best_mse:
            best_mse = final_mse
            best_history = list(env.history)
        
        if (ep + 1) % 60 == 0:
            avg_r = np.mean(episode_rewards[-60:])
            avg_mse = np.mean(episode_mses[-60:])
            print(f'  [{target_name}] Ep {ep+1}/{n_episodes} | '
                  f'Avg Reward: {avg_r:.3f} | Avg MSE: {avg_mse:.4f} | '
                  f'Best MSE: {best_mse:.4f}')
    
    return agent, episode_rewards, episode_mses, best_history


print('Training painting agents...')
results = {}
for name, target in targets.items():
    print(f'\nTraining on "{name}"...')
    agent, rewards, mses, history = train_painter(target, name, n_episodes=300)
    results[name] = {
        'agent': agent, 'rewards': rewards,
        'mses': mses, 'history': history
    }

print('\nAll agents trained!')

## 6. Training Curves

In [ ]:
def smooth(values, window=30):
    return [np.mean(values[max(0,i-window):i+1]) for i in range(len(values))]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors_map = {'Red Circle': '#e74c3c', 'Gradient': '#3498db',
              'Flag': '#e67e22', 'Green Cross': '#2ecc71'}

for name, res in results.items():
    c = colors_map[name]
    axes[0].plot(smooth(res['rewards']), label=name, color=c, linewidth=2)
    axes[1].plot(smooth(res['mses']), label=name, color=c, linewidth=2)

axes[0].set_title('Episode Rewards', fontweight='bold', fontsize=13)
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Total Reward')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_title('Reconstruction MSE', fontweight='bold', fontsize=13)
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('MSE')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Painting Agent Training Progress', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Step-by-Step Painting Visualization

Watch the agent "paint" each target image stroke by stroke.

In [ ]:
for name, target in targets.items():
    history = results[name]['history']
    if history is None:
        continue
    
    n_snapshots = min(len(history), 10)
    indices = np.linspace(0, len(history) - 1, n_snapshots, dtype=int)
    
    fig, axes = plt.subplots(1, n_snapshots + 1, figsize=(3 * (n_snapshots + 1), 3))
    
    for i, idx in enumerate(indices):
        axes[i].imshow(history[idx].transpose(1, 2, 0), interpolation='nearest')
        if idx == 0:
            axes[i].set_title('Start', fontsize=9, fontweight='bold')
        else:
            axes[i].set_title(f'Stroke {idx}', fontsize=9)
        axes[i].axis('off')
    
    axes[-1].imshow(target.transpose(1, 2, 0), interpolation='nearest')
    axes[-1].set_title('Target', fontsize=9, fontweight='bold')
    axes[-1].axis('off')
    
    plt.suptitle(f'Painting Progression: {name}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 8. Final Results: Target vs Agent's Painting

In [ ]:
fig, axes = plt.subplots(2, len(targets), figsize=(4 * len(targets), 8))

for i, (name, target) in enumerate(targets.items()):
    axes[0, i].imshow(target.transpose(1, 2, 0), interpolation='nearest')
    axes[0, i].set_title(f'Target: {name}', fontsize=11, fontweight='bold')
    axes[0, i].axis('off')
    
    history = results[name]['history']
    final = history[-1] if history else np.full_like(target, 0.5)
    axes[1, i].imshow(final.transpose(1, 2, 0), interpolation='nearest')
    mse = np.mean((final - target) ** 2)
    axes[1, i].set_title(f'Agent (MSE: {mse:.4f})', fontsize=11)
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Target', fontsize=13, fontweight='bold')
axes[1, 0].set_ylabel('Agent\'s Painting', fontsize=13, fontweight='bold')

plt.suptitle('Target vs Agent\'s Painting', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Different "Painting Styles" from Different Reward Weightings

By adjusting the reward parameters, we produce different artistic behaviors:
- **Detailed**: high improvement weight, low stroke penalty
- **Minimalist**: low improvement weight, high stroke penalty
- **Impressionist**: medium weights, larger brush sizes preferred

In [ ]:
target = targets['Red Circle']

styles = {
    'Detailed': {'stroke_penalty': 0.0002, 'improve_scale': 800.0, 'max_strokes': 60},
    'Balanced': {'stroke_penalty': 0.001, 'improve_scale': 500.0, 'max_strokes': 50},
    'Minimalist': {'stroke_penalty': 0.01, 'improve_scale': 300.0, 'max_strokes': 25},
}

style_results = {}
print('Training different painting styles...')
for style_name, params in styles.items():
    print(f'\n  Style: {style_name} (penalty={params["stroke_penalty"]}, '
          f'scale={params["improve_scale"]}, strokes={params["max_strokes"]})')
    agent, rewards, mses, history = train_painter(
        target, f'{style_name}', n_episodes=200, lr=1e-3, **params
    )
    style_results[style_name] = {'history': history, 'mses': mses}

print('\nAll styles trained!')

In [ ]:
fig, axes = plt.subplots(len(styles) + 1, 8, figsize=(24, 3 * (len(styles) + 1)))

for ax_row in axes:
    for ax in ax_row:
        ax.axis('off')

# Target row
for j in range(8):
    axes[0, j].imshow(target.transpose(1, 2, 0), interpolation='nearest')
axes[0, 0].set_title('Target', fontsize=11, fontweight='bold')
for j in range(1, 8):
    axes[0, j].set_visible(False)
axes[0, 0].set_ylabel('Target', fontsize=12, fontweight='bold', rotation=0, labelpad=60)

for row_idx, (style_name, res) in enumerate(style_results.items(), 1):
    history = res['history']
    if history is None:
        continue
    n_show = min(len(history), 8)
    indices = np.linspace(0, len(history) - 1, n_show, dtype=int)
    
    for j, idx in enumerate(indices):
        axes[row_idx, j].imshow(history[idx].transpose(1, 2, 0), interpolation='nearest')
        axes[row_idx, j].axis('off')
        if idx == 0:
            axes[row_idx, j].set_title('Start', fontsize=9)
        elif idx == len(history) - 1:
            axes[row_idx, j].set_title('Final', fontsize=9, fontweight='bold')
        else:
            axes[row_idx, j].set_title(f'Stroke {idx}', fontsize=9)
    
    axes[row_idx, 0].set_ylabel(style_name, fontsize=12, fontweight='bold', rotation=0, labelpad=60)

plt.suptitle('Painting Styles: Different Reward Weightings', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(styles) + 1, figsize=(4 * (len(styles) + 1), 4))

axes[0].imshow(target.transpose(1, 2, 0), interpolation='nearest')
axes[0].set_title('Target', fontsize=12, fontweight='bold')
axes[0].axis('off')

style_colors = {'Detailed': '#2ecc71', 'Balanced': '#3498db', 'Minimalist': '#e74c3c'}

for i, (style_name, res) in enumerate(style_results.items(), 1):
    history = res['history']
    final = history[-1] if history else np.full_like(target, 0.5)
    mse = np.mean((final - target) ** 2)
    n_strokes = len(history) - 1
    
    axes[i].imshow(final.transpose(1, 2, 0), interpolation='nearest')
    axes[i].set_title(f'{style_name}\n{n_strokes} strokes, MSE={mse:.4f}',
                      fontsize=11, fontweight='bold')
    axes[i].axis('off')

plt.suptitle('Painting Style Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for style_name, res in style_results.items():
    ax.plot(smooth(res['mses'], 20), label=style_name,
            color=style_colors.get(style_name, 'gray'), linewidth=2)

ax.set_title('MSE Convergence by Painting Style', fontsize=14, fontweight='bold')
ax.set_xlabel('Episode', fontsize=12)
ax.set_ylabel('Final MSE', fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Analysis

### What the Agent Learns

The painting agent discovers several strategies:

1. **Background first**: Large strokes to establish the dominant color early, reducing MSE quickly
2. **Foreground details**: Smaller, targeted strokes for precise features (e.g., circle edges)
3. **Color matching**: Learning which discretized colors best approximate the target

### Style Emergence

The reward weighting directly shapes the agent's behavior:

| Style | Penalty | Scale | Effect |
|---|---|---|---|
| Detailed | Low | High | Many small strokes, high fidelity |
| Balanced | Medium | Medium | Good quality with reasonable stroke count |
| Minimalist | High | Low | Few bold strokes, captures essence |

This mirrors real artistic styles: photorealism (detailed) vs impressionism (minimalist).

### Mathematical Insight

The optimal policy balances the marginal improvement of each stroke against its cost:

$$\pi^*(a|s) = \arg\max_a \left[\lambda_{\text{improve}} \cdot \Delta\text{MSE}(s, a) - \lambda_{\text{stroke}}\right]$$

When $\lambda_{\text{stroke}} \gg \lambda_{\text{improve}} \cdot \max_a \Delta\text{MSE}$, the agent prefers to stop early, creating minimalist art.

## Summary

In this notebook, we built an RL agent that learns to paint target images stroke by stroke:

1. **MDP formulation**: State captures canvas statistics, actions specify brush strokes (position, color, size), and rewards balance reconstruction quality against efficiency

2. **Factored action space**: Instead of a monolithic action space, separate policy heads for position, color, and size make learning tractable

3. **REINFORCE training**: The agent learns to place strokes strategically — background first, then details — through trial-and-error optimization

4. **Painting styles**: Different reward weightings (improvement scale vs stroke penalty) produce naturally distinct artistic styles, from detailed to minimalist

5. **Creative potential**: This framework extends to more expressive brush models (curves, textures), larger canvases, and real image targets

### Next Steps

- Add more brush types (rectangles, curves, spray)
- Use a CNN-based policy that directly processes the canvas and target images
- Scale to larger images with hierarchical (coarse-to-fine) painting
- Add a discriminator-based reward for more perceptual quality